## **Data Cleaning & Data Quality Validation**

**Tujuan Proyek**

Melakukan pemeriksaan dan pembersihan data OULAD untuk memastikan kualitas, konsistensi, dan kesiapan dataset sebelum digunakan dalam analisis aktivitas pembelajaran serta hasil akhir mahasiswa.

**Tujuan Pembersihan Data**

1. Memeriksa nilai yang hilang dan nilai khusus yang merepresentasikan data kosong.

2. Mengidentifikasi duplikasi data.

3. Memvalidasi tipe data dan nilai kategorikal.

4. Memeriksa konsistensi hubungan antar-dataset.

5. Menyimpan data hasil pembersihan untuk tahap analisis berikutnya.

#### **Mengimpor Library dan Memuat Dataset**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

base_path = Path(
    "/content/drive/MyDrive/Data_Analytics/"
    "edtech-learning-analytics/data/raw"
)

student_info = pd.read_csv(base_path / "studentInfo.csv")
student_vle = pd.read_csv(base_path / "studentVle.csv")
student_registration = pd.read_csv(base_path / "studentRegistration.csv")
courses = pd.read_csv(base_path / "courses.csv")

print("Dataset berhasil dimuat")

Mounted at /content/drive
Dataset berhasil dimuat


#### **Memeriksa Nilai yang Hilang**

In [ ]:
datasets = {
    "student_info": student_info,
    "student_vle": student_vle,
    "student_registration": student_registration,
    "courses": courses
}

for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print(df.isnull().sum())


===== student_info =====
code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64

===== student_vle =====
code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

===== student_registration =====
code_module            0
code_presentation      0
id_student             0
date_registration      0
date_unregistration    0
dtype: int64

===== courses =====
code_module                   0
code_presentation             0
module_presentation_length    0
dtype: int64


#### **Memeriksa Nilai Khusus**

In [ ]:
print("Date Registration:")
print(student_registration["date_registration"].value_counts().head(10))

print("\nDate Unregistration:")
print(student_registration["date_unregistration"].value_counts().head(10))

print("\nJumlah nilai '?':")
print((student_registration[["date_registration", "date_unregistration"]] == "?").sum())

Date Registration:
date_registration
-22    1034
-29     982
-24     741
-25     730
-23     729
-30     581
-36     553
-28     548
-32     497
-17     488
Name: count, dtype: int64

Date Unregistration:
date_unregistration
?     22521
12      788
0       419
27      170
-1      147
-2       99
-7       99
10       91
-5       90
-8       87
Name: count, dtype: int64

Jumlah nilai '?':
date_registration         45
date_unregistration    22521
dtype: int64


#### **Memeriksa Data Duplikat**

In [ ]:
# Pemeriksaan duplikasi seluruh baris
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print("Jumlah duplikasi:", df.duplicated().sum())


===== student_info =====
Jumlah duplikasi: 0

===== student_vle =====
Jumlah duplikasi: 787170

===== student_registration =====
Jumlah duplikasi: 0

===== courses =====
Jumlah duplikasi: 0


In [ ]:
# Pemeriksaan duplikasi berdasarkan key
student_key = ["code_module", "code_presentation", "id_student"]

print("Duplikasi studentInfo:", student_info.duplicated(subset=student_key).sum())

print("Duplikasi studentRegistration:", student_registration.duplicated(subset=student_key).sum())

Duplikasi studentInfo: 0
Duplikasi studentRegistration: 0


#### **Memvalidasi Tipe Data**

In [ ]:
# Pemeriksaan tipe data
for name, df in datasets.items():
  print(f"\n===== {name} =====")
  print(df.dtypes)


===== student_info =====
code_module             object
code_presentation       object
id_student               int64
gender                  object
region                  object
highest_education       object
imd_band                object
age_band                object
num_of_prev_attempts     int64
studied_credits          int64
disability              object
final_result            object
dtype: object

===== student_vle =====
code_module          object
code_presentation    object
id_student            int64
id_site               int64
date                  int64
sum_click             int64
dtype: object

===== student_registration =====
code_module            object
code_presentation      object
id_student              int64
date_registration      object
date_unregistration    object
dtype: object

===== courses =====
code_module                   object
code_presentation             object
module_presentation_length     int64
dtype: object


In [ ]:
# Membersihkan nilai tanggal
date_columns = ["date_registration", "date_unregistration"]

for col in date_columns:
    student_registration[col] = pd.to_numeric(student_registration[col], errors="coerce")

print(student_registration[date_columns].dtypes)

print("\nMissing values setelah konversi:")
print(student_registration[date_columns].isnull().sum())

date_registration      float64
date_unregistration    float64
dtype: object

Missing values setelah konversi:
date_registration         45
date_unregistration    22521
dtype: int64


#### **Memvalidasi Nilai Kategorikal**

In [ ]:
# Memeriksa konsistensi nilai pada kolom kategorikal
categorical_columns = ["gender", "highest_education", "age_band", "disability", "final_result"]

for col in categorical_columns:
    print(f"\n===== {col} =====")
    print(student_info[col].value_counts(dropna=False))


===== gender =====
gender
M    17875
F    14718
Name: count, dtype: int64

===== highest_education =====
highest_education
A Level or Equivalent          14045
Lower Than A Level             13158
HE Qualification                4730
No Formal quals                  347
Post Graduate Qualification      313
Name: count, dtype: int64

===== age_band =====
age_band
0-35     22944
35-55     9433
55<=       216
Name: count, dtype: int64

===== disability =====
disability
N    29429
Y     3164
Name: count, dtype: int64

===== final_result =====
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64


In [ ]:
# Memeriksa apakah terdapat nilai yang tidak valid pada kolom final_result
valid_results = ["Pass", "Distinction", "Fail", "Withdrawn"]

invalid_results = student_info[
    ~student_info["final_result"].isin(valid_results)
]

print("Jumlah final_result tidak valid:", len(invalid_results))

Jumlah final_result tidak valid: 0


#### **Memvalidasi Nilai Numerik**

In [ ]:
print("Jumlah sum_click negatif:")
print((student_vle["sum_click"] < 0).sum())

print("\nStatistik sum_click:")
print(student_vle["sum_click"].describe())

print("\nJumlah studied_credits negatif:")
print((student_info["studied_credits"] < 0).sum())

Jumlah sum_click negatif:
0

Statistik sum_click:
count    1.065528e+07
mean     3.716946e+00
std      8.849047e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      6.977000e+03
Name: sum_click, dtype: float64

Jumlah studied_credits negatif:
0


#### **Validasi Konsistensi Data**

In [ ]:
# Validasi Konsistensi Status Registrasi
registration_check = student_info[["code_module", "code_presentation", "id_student", "final_result"]].merge(student_registration[["code_module", "code_presentation", "id_student", "date_unregistration"]],
    on=["code_module", "code_presentation", "id_student"],
    how="left",
    validate="one_to_one"
)

print(
    pd.crosstab(
        registration_check["final_result"],
        registration_check["date_unregistration"].isna()
    )
)

date_unregistration  False  True 
final_result                     
Distinction              0   3024
Fail                     9   7043
Pass                     0  12361
Withdrawn            10063     93


In [ ]:
# Validasi duplikasi student_vle berdasarkan key
vle_key = ["code_module", "code_presentation", "id_student", "id_site", "date"]

print("Duplikasi berdasarkan key:")
print(student_vle.duplicated(subset=vle_key).sum())

print("\nContoh baris duplikat:")
print(student_vle[student_vle.duplicated(keep=False)].head(10))

Duplikasi berdasarkan key:
2195960

Contoh baris duplikat:
    code_module code_presentation  id_student  id_site  date  sum_click
1           AAA             2013J       28400   546652   -10          1
2           AAA             2013J       28400   546652   -10          1
62          AAA             2013J       45462   546652   -10          1
63          AAA             2013J       45462   546652   -10          1
178         AAA             2013J       77367   546652   -10          4
180         AAA             2013J       77367   546652   -10          4
191         AAA             2013J       94961   546652   -10          2
193         AAA             2013J       94961   546652   -10          2
432         AAA             2013J      248270   546652   -10          4
442         AAA             2013J      248270   546652   -10          4


In [ ]:
# Validasi Variasi Klik student_vle
vle_key = ["code_module", "code_presentation", "id_student", "id_site", "date"]

click_variation = (student_vle.groupby(vle_key)["sum_click"].nunique())

print("Jumlah kombinasi key dengan variasi sum_click:")
print((click_variation > 1).sum())

print("\nJumlah kombinasi key dengan satu nilai sum_click:")
print((click_variation == 1).sum())

Jumlah kombinasi key dengan variasi sum_click:
1179074

Jumlah kombinasi key dengan satu nilai sum_click:
7280246


In [ ]:
# Validasi Hubungan student_vle dengan student_info
student_key = ["code_module", "code_presentation", "id_student"]

vle_students = student_vle[student_key].drop_duplicates()

info_students = student_info[student_key].drop_duplicates()

validation = vle_students.merge(info_students, on=student_key, how="left", indicator=True)

print(validation["_merge"].value_counts())

_merge
both          29228
left_only         0
right_only        0
Name: count, dtype: int64


#### **Menyimpan Dataset yang Telah Dibersihkan**

In [ ]:
processed_path = Path(
    "/content/drive/MyDrive/Data_Analytics/"
    "edtech-learning-analytics/data/processed"
)

processed_path.mkdir(parents=True, exist_ok=True)

student_info.to_csv(processed_path / "studentInfo_clean.csv", index=False)

student_vle.to_csv(processed_path / "studentVle_clean.csv", index=False)

student_registration.to_csv(processed_path / "studentRegistration_clean.csv", index=False)

courses.to_csv(processed_path / "courses_clean.csv", index=False)

print("Semua dataset berhasil disimpan!")

Semua dataset berhasil disimpan!


## **Dokumentasi**

**1. Penilaian Missing Values**

Pemeriksaan awal menggunakan isnull().sum() menunjukkan bahwa keempat dataset tidak memiliki missing values yang terdeteksi secara langsung.

Setelah kolom tanggal pada studentRegistration dikonversi menjadi numerik, ditemukan:

- date_registration: 45 missing values.
- date_unregistration: 22.521 missing values.

Nilai kosong pada date_unregistration tidak langsung dianggap sebagai kesalahan karena dapat berkaitan dengan mahasiswa yang tidak mengundurkan diri. Missing values tetap dipertahankan sebagai NaN untuk menghindari perubahan makna data.

**2. Penilaian Data Duplikat**

Hasil pemeriksaan duplikasi seluruh baris:

- studentInfo: 0 duplikasi.
- studentVle: 787.170 duplikasi.
- studentRegistration: 0 duplikasi.
- courses: 0 duplikasi.

Pemeriksaan tambahan pada studentVle berdasarkan kombinasi code_module, code_presentation, id_student, id_site, dan date menemukan 2.195.960 baris duplikat berdasarkan key.

Hasil tersebut menunjukkan adanya beberapa catatan aktivitas pada kombinasi mahasiswa, materi pembelajaran, dan hari yang sama. Namun, catatan tersebut belum dapat dipastikan sebagai kesalahan pencatatan.

**Keputusan pembersihan data:** Seluruh catatan studentVle dipertahankan tanpa penghapusan duplikasi karena belum terdapat bukti yang cukup untuk membedakan aktivitas berulang yang valid dengan kesalahan pencatatan.

**3. Validasi Tipe Data**

Kolom date_registration dan date_unregistration dikonversi dari tipe object menjadi numerik menggunakan pd.to_numeric() karena nilainya merepresentasikan jumlah hari relatif terhadap awal periode pembelajaran.

Nilai yang tidak dapat dikonversi menjadi angka diubah menjadi NaN menggunakan errors="coerce".

**4. Validasi Konsistensi Data**

**a. Validasi Konsistensi Status Registrasi**

Pemeriksaan hubungan antara final_result dan date_unregistration menunjukkan:

- Sebanyak 10.063 dari 10.156 observasi berstatus Withdrawn memiliki tanggal pengunduran diri (99,08%).
- Terdapat 93 observasi Withdrawn tanpa tanggal pengunduran diri.
- Terdapat 9 observasi Fail yang memiliki tanggal pengunduran diri.
- Seluruh observasi Pass dan Distinction tidak memiliki tanggal pengunduran diri.

Sebanyak 102 observasi memerlukan perhatian lebih lanjut karena terdapat ketidaklengkapan atau ketidaksesuaian informasi pengunduran diri.

Data tersebut tetap dipertahankan, dan final_result digunakan sebagai acuan utama dalam analisis hasil pembelajaran.

**b. Validasi duplikasi student_vle berdasarkan key**

Pemeriksaan berdasarkan kombinasi mahasiswa, modul, materi, dan hari menemukan 2.195.960 baris yang memiliki key sama dengan baris sebelumnya.

Catatan aktivitas tersebut tetap dipertahankan karena kesamaan key belum cukup untuk membuktikan adanya kesalahan data.

**c. Validasi Variasi Klik student_vle**

Pemeriksaan dilakukan untuk mengetahui apakah kombinasi key yang sama memiliki variasi nilai sum_click.

- Kombinasi key dengan variasi sum_click: 1.179.074.
- Kombinasi key dengan satu nilai sum_click: 7.280.246.

Hasil pemeriksaan digunakan untuk memahami karakteristik pencatatan aktivitas pada dataset studentVle.

**d. Validasi Hubungan Data Student VLE dan Student Info**

Pemeriksaan hubungan berdasarkan kombinasi code_module, code_presentation, dan id_student menghasilkan:

- both: 29.228.
- left_only: 0.
- right_only: 0 (tidak ditemukan karena menggunakan left join).

Hasil tersebut menunjukkan bahwa seluruh kombinasi mahasiswa-modul-periode yang memiliki aktivitas pada studentVle ditemukan dalam studentInfo.

**5. Ringkasan Pembersihan Data**

Berdasarkan pemeriksaan kualitas data, ditemukan missing values pada kolom tanggal, duplikasi catatan aktivitas pada studentVle, serta sejumlah observasi dengan informasi pengunduran diri yang tidak lengkap atau perlu ditinjau.

Proses cleaning yang dilakukan meliputi konversi tipe data tanggal menjadi numerik dan validasi kualitas serta konsistensi data.

Missing values dan duplikasi aktivitas dipertahankan karena belum terdapat bukti yang cukup untuk melakukan penghapusan atau perubahan nilai.

Dataset disiapkan untuk tahap analisis berikutnya dengan tetap mempertimbangkan keterbatasan kualitas data yang ditemukan.
